# 5. Where do documents begin and end?

Every notebook so far has worked with paragraphs whose boundaries we already knew &mdash; each row in our dataset comes with a `resolution_id` telling us exactly which resolution it belongs to. Many real archives aren't this convenient. Sometimes all you have is one long, continuous transcription of a scanned book, with no markers at all for where one document ends and the next begins.

This notebook asks: can the formulaic phrases we've already found help us figure out where those boundaries are, *without* being told in advance?

## Testing this honestly: hide the answer key, then check our work

To find out whether a boundary-finding method actually works, we need to be able to check its guesses against the truth. Conveniently, our example dataset *does* contain the true boundaries (via `resolution_id`) &mdash; so for this notebook, we'll deliberately set that information aside while making our guesses, using only the words themselves, the same way we would if we genuinely didn't have it. Only afterwards will we use `resolution_id` to grade ourselves.

Let's load the data and build two things: one continuous stream of tokens (pretending we don't know where the documents are), and, secretly, the list of token positions where a new `resolution_id` actually begins &mdash; our answer key, which we'll only look at when checking our guesses.

In [1]:
import csv
import gzip
import re

TOKEN_PATTERN = re.compile(r"[A-Za-z\u00c0-\u00ff]+")


def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())


rows = []
with gzip.open('data/resolution_paragraphs/paragraphs-3823.tsv.gz', 'rt', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        rows.append(row)

flat_stream = []
true_starts = []  # the answer key -- not used for guessing, only for grading
previous_resolution_id = None
offset = 0
for row in rows:
    tokens = tokenize(row['text'])
    if row['resolution_id'] != previous_resolution_id:
        true_starts.append(offset)
        previous_resolution_id = row['resolution_id']
    flat_stream.extend(tokens)
    offset += len(tokens)

print(f'{len(flat_stream)} tokens in one continuous stream')
print(f'{len(true_starts)} true document starts (kept aside, for grading only)')

449905 tokens in one continuous stream
2296 true document starts (kept aside, for grading only)


## The core idea: a reliable opener marks a likely start

We already know, from [notebook 1](01-what-is-a-formula.ipynb) and [notebook 3](03-finding-repeated-phrases.ipynb), that many resolutions open with the formula `ontfangen een missive van`. If that formula reliably appears at the *start* of a document and essentially nowhere else, then **every place it occurs is itself a guess at where a document begins**. Let's try exactly that, with nothing more sophisticated than searching for this one phrase across the whole stream.

In [2]:
def find_occurrences(stream, phrase_tokens):
    n = len(phrase_tokens)
    return [i for i in range(len(stream) - n + 1) if stream[i:i + n] == phrase_tokens]


guessed_starts = find_occurrences(flat_stream, 'ontfangen een missive van'.split())
print(f'{len(guessed_starts)} guessed document starts')

1267 guessed document starts


## Checking our guesses

Now we reveal the answer key and compare. Two questions matter, and they're different questions:

* **Precision** &mdash; of all the guesses we made, what fraction were actually right? ("When we said *start here*, how often were we correct?")
* **Recall** &mdash; of all the true document starts, what fraction did we actually find? ("Of everything we were looking for, how much did we catch?")

A method can score well on one and poorly on the other &mdash; guessing "start" at *every single word* would catch every true start (perfect recall) while being right almost none of the time (terrible precision). We need both numbers to know whether a method is actually useful.

We'll count a guess as correct if it lands within 2 tokens of a true start, to allow for tiny differences in exactly which token counts as "the" start.

In [3]:
def evaluate(guessed, true, tolerance=2):
    correct_guesses = sum(1 for g in guessed if any(abs(g - t) <= tolerance for t in true))
    found_true = sum(1 for t in true if any(abs(g - t) <= tolerance for g in guessed))
    precision = correct_guesses / len(guessed)
    recall = found_true / len(true)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1


precision, recall, f1 = evaluate(guessed_starts, true_starts)
print(f'precision = {precision:.2f}   (when we guessed "start", we were right {precision:.0%} of the time)')
print(f'recall    = {recall:.2f}   (we found {recall:.0%} of all the true document starts)')

precision = 0.99   (when we guessed "start", we were right 99% of the time)
recall    = 0.55   (we found 55% of all the true document starts)


(The third number, **F1**, is a single combined score balancing precision and recall, useful for comparing methods at a glance &mdash; we'll print it alongside the other two from here on.)

Already a useful result from one phrase: when this formula occurs, it's almost always genuinely a document start. But it only catches a little over half of the true starts &mdash; this corpus has more than one kind of document, and not all of them open with "received a letter from...". Let's add a second known opener, for resolutions that respond to a petition rather than a letter.

In [4]:
opener_phrases = ['ontfangen een missive van', 'is ter vergaderinge geleesen de requeste van']

guessed_starts = set()
for phrase in opener_phrases:
    guessed_starts.update(find_occurrences(flat_stream, phrase.split()))
guessed_starts = sorted(guessed_starts)

precision, recall, f1 = evaluate(guessed_starts, true_starts)
print(f'{len(guessed_starts)} guesses   precision = {precision:.2f}   recall = {recall:.2f}   f1 = {f1:.2f}')

1598 guesses   precision = 0.99   recall = 0.69   f1 = 0.82


Recall improved noticeably, for free, just by knowing about one more opener. Precision stayed just as high. Let's look at what's still going wrong, on both sides, before deciding whether to add anything else.

In [5]:
wrong_guesses = [g for g in guessed_starts if not any(abs(g - t) <= 2 for t in true_starts)]
missed_starts = [t for t in true_starts if not any(abs(g - t) <= 2 for g in guessed_starts)]

print(f'{len(wrong_guesses)} wrong guesses out of {len(guessed_starts)}')
for w in wrong_guesses[:2]:
    print('  ...', ' '.join(flat_stream[max(0, w - 6):w + 8]), '...')

print()
print(f'{len(missed_starts)} true starts we missed out of {len(true_starts)}')
for m in missed_starts[:2]:
    print('  ...', ' '.join(flat_stream[max(0, m - 2):m + 10]), '...')

8 wrong guesses out of 1598
  ... de anno saemiz veneris den januars ontfangen een missive van den heere grave van ...
  ... vergaderinge rapport te doen den febrnary ontfangen een missive van het collegie ter admiraliteit ...

706 true starts we missed out of 2296
  ... luna den january praeside den heere van palland tot olinthuys ...
  ... worden beantwoord ynde ter vergaderinge getoont en geexhibeert de instructien van de ...


The handful of wrong guesses are close calls &mdash; genuine occurrences of our opener phrase that just happen to land slightly differently from where we drew the "official" boundary (often because of a short attendance-list entry sitting in between two resolutions). The missed starts tell a clearer story: paragraphs that begin with `luna den ...`, `martis den ...`, and similar &mdash; the Latin weekday names that open each day's *attendance list*, a different kind of document entirely, with its own opener that we haven't told the software about yet. We saw these same weekday openers back in the [scale-grouping notebook](../Phrase-Recurrence-Scale-Grouping.ipynb); let's add them here too.

In [6]:
day_names = ['luna den', 'martis den', 'mercurii den', 'jovis den', 'veneris den', 'sabbathi den']

for phrase in day_names:
    guessed_starts.extend(find_occurrences(flat_stream, phrase.split()))
guessed_starts = sorted(set(guessed_starts))

precision, recall, f1 = evaluate(guessed_starts, true_starts)
print(f'{len(guessed_starts)} guesses   precision = {precision:.2f}   recall = {recall:.2f}   f1 = {f1:.2f}')

1859 guesses   precision = 0.99   recall = 0.80   f1 = 0.88


Recall climbs again, with precision still essentially untouched. The pattern across all three attempts is the real lesson of this notebook:

> **Finding document boundaries well isn't about one clever trick &mdash; it's about systematically collecting a good opener for every *kind* of document in your corpus.** Each time we noticed a kind of document our list didn't cover yet, and added its opener, recall went up while precision stayed high.

That's a more useful, more general insight than it might first appear: it tells you *where to look* when a boundary-finding attempt isn't working well &mdash; not at the maths, but at what kinds of documents you haven't found an opener for yet. [Notebook 6](06-naming-your-formulas.ipynb) gives you an interactive tool for exactly that: systematically going through the formulas you've found, organising them by what kind of document or document part they belong to, and spotting which kinds you might be missing.

## A note on more advanced approaches

The toolkit also includes more advanced machinery for boundary-finding: automatically discovering which *sets* of phrases tend to occur together in a fixed order (called **motifs**), and using those patterns to score likely boundaries, without you first having to identify good opener phrases by hand. This sounds like it should outperform the simple approach in this notebook &mdash; in practice, when we tried it on a real corpus like this one, it often did *worse*, because automatically ranking candidate patterns tends to favour rare, narrowly-specific combinations over the few common, reliable openers that actually mark most boundaries.

The lesson isn't "never use the more advanced tools" &mdash; it's that more machinery doesn't automatically mean a better result, and it's always worth comparing against the simplest approach that could plausibly work, the way we built up our answer through this notebook, before reaching for something more complex. If you do want to explore the more advanced approach, it's covered in [`Motif-Based-Boundary-Detection.ipynb`](../Motif-Based-Boundary-Detection.ipynb) and discussed honestly, warts and all, in [`Document-Pattern-Detection-Findings.md`](../Document-Pattern-Detection-Findings.md).

## Next steps

We've now seen, several times over, that this corpus contains different *kinds* of formulas serving different roles &mdash; openers for letters, openers for petitions, openers for attendance lists, decision formulas, and so on. [Notebook 6](06-naming-your-formulas.ipynb) introduces an interactive tool for organising the formulas you find into categories like these yourself, browsing real examples as you go, rather than keeping track of everything by hand as we've been doing in this notebook.